# DRC CoRL 2026 — Smoke Test (CPU, no GPU needed)

Run this **first**, on a CPU kernel, to confirm the whole pipeline works on Kaggle before
spending GPU quota. It pulls the code from GitHub, installs the dev dependencies, and runs:
1. **preflight** — 9-point readiness gate
2. **Part A** controlled validation (real theory results: bound tightness, identifiability, horizon wall)
3. **smoke test** — full synthetic train→metrics→rollouts→analysis for both architectures
4. **pytest** — unit + theorem-validation tests

No accelerator required. Set `REPO_URL` to your fork.

In [ ]:
# --- pull code from GitHub ---
REPO_URL = "https://github.com/YOUR_USERNAME/CORL.git"   # <-- set this
BRANCH   = "main"
%cd /kaggle/working
![ -d CORL ] && (cd CORL && git pull) || git clone -b $BRANCH $REPO_URL CORL
%cd /kaggle/working/CORL
!git log --oneline -1

In [ ]:
# --- dev dependencies only (no simulator needed for the smoke test) ---
!pip install -q numpy scipy pandas scikit-learn statsmodels matplotlib PyYAML torch pytest pyflakes

In [ ]:
# 1. Preflight readiness gate (config consistency, adapter routing, both architectures, ...)
!python scripts/preflight.py

In [ ]:
# 2. Part A controlled validation — REAL theory results (CPU, seconds)
#    P1 bound tightness (R^2=1), P2 identifiability (0.31 vs 0.84), selection regret, horizon wall.
!python scripts/run_partA_bound.py

In [ ]:
# show the Part A figures inline
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figures/partA/*.pdf')):
    print(f)
# (PDFs; convert to PNG to preview if desired)
import json; print(json.dumps(json.load(open('results/partA_bound.json')), indent=2)[:800])

In [ ]:
# 3. Full synthetic pipeline, BOTH architectures (train -> 8 metrics -> rollouts -> H1-H4)
!python scripts/smoke_test.py

In [ ]:
# 4. Unit + theorem-validation tests
!python -m pytest -q

If all four pass, the code is healthy on Kaggle. Proceed to `01_session1_libero.ipynb`
(switch the accelerator to **GPU T4 x2**).